In [2]:
import pandas as pd
import os


In [3]:
basePath = '5YearSpanDatasets'
forces = ['metropolitan', 'thames', 'surrey', 'sussex']

allData = []

# Checks each folder in the updatedDatasets folder
for folder in os.listdir(basePath):
    # Concatenates the folder name to the updatedDatasets to create a new file path
    filePath = os.path.join(basePath, folder)
    # Checks if the folder actually exists
    if os.path.isdir(filePath):
        # Checks each file in the file path
        for file in os.listdir(filePath):
            # Checks the file contains one of the police forces in the filename
            if any(force in file.lower() for force in forces):
                # Updates file path to include the folder
               file_path = os.path.join(filePath, file)
               
               df=pd.read_csv(file_path)
               df['Month'] = folder
               allData.append(df)
    

               


In [4]:
if allData:
    masterDf = pd.concat(allData, ignore_index=True)
else:
    print("No data found to process")

In [5]:
# Shows number of records in the master data
print(f"Before preprocessing, there are {len(masterDf)} values in the data")

# Shows each column and if they have any null values
print(f"\nList of null values in every column \n {masterDf.isnull().sum()}\n")

# Shows the unique values for each column
for col in masterDf:
    print(f" - The unique values for {col} are {masterDf[col].unique()}")


Before preprocessing, there are 9453125 values in the data

List of null values in every column 
 Crime ID                 2111038
Month                          0
Reported by                    0
Falls within                   0
Longitude                 176614
Latitude                  176614
Location                       0
LSOA code                 176644
LSOA name                 176644
Crime type                     0
Last outcome category    2111038
Context                  9453125
dtype: int64

 - The unique values for Crime ID are ['a8977a2a4e14252420371eb993d52e4d0b8288a7c833e66ba79592b3e2b982e7' nan
 '934e173f2bc2e1dd3a257b37939d8f97575d3eeb89ff0c3eee2c73f16630edc5' ...
 'cb80080e0164fa9d12767bcfe912ed902b1bc3db0ce83d50b7f6bc055d313392'
 'c49ac623430308ffb6f815e12725a9d3244c7bf3275592a22953ed4f5964c326'
 '14fb64f2b5de925c7c1cec2234f64d221dada2af05b6bc07fe41db9139e021ea']
 - The unique values for Month are ['2019-01' '2019-02' '2019-03' '2019-04' '2019-05' '2019-06' '2019-07'

In [6]:
totalBeforeDupRemoval = len(masterDf)
print(f"There are {totalBeforeDupRemoval} values in the dataset before any processing at all")
duplicateCount = masterDf.duplicated().sum()
print(f"There are {duplicateCount} duplicate rows.")
masterDf = masterDf.drop_duplicates()
totalAfterDupRemoval = len(masterDf)
print(f"There are {totalAfterDupRemoval} values in the dataset after removing duplicates")

There are 9453125 values in the dataset before any processing at all
There are 1011718 duplicate rows.
There are 8441407 values in the dataset after removing duplicates


In [7]:
# Handling Missing Values
# Missing values in Crime ID , Longitude, Latitude, LSOA Code, LSOA Name, Last outcome category, Context
# Context is missing for every single record so can be dropped

print(f"There are {totalAfterDupRemoval} total records")

# Shows each column and if they have any null values
print(f"\nList of null values in every column before handling null values\n {masterDf.isnull().sum()}\n")

# Crime ID: add placeholder value "No Crime ID"
# Longitude and Latitude: add placeholder value 0
# LSOA code: add placeholder value "No LSOA Code Assigned"
# LSOA name: add placeholder value "No LSOA name Assigned"
# Last outcome category: add placeholder value "Unknown outcome category"
# Context: drop since every record has null values in context

placeholders = {
    'Crime ID': 'No Crime ID',
    'Longitude': 0,
    'Latitude' : 0,
    'LSOA code' : 'No LSOA Code assigned',
    'LSOA name' : 'No LSOA Name assigned',
    'Last outcome category' : 'Unknown outcome category'
}

masterDf = masterDf.fillna(value=placeholders)
masterDf = masterDf.drop("Context", axis='columns')

# Shows each column and if they have any null values
print(f"\nList of null values in every column \n {masterDf.isnull().sum()}\n")

There are 8441407 total records

List of null values in every column before handling null values
 Crime ID                 1155292
Month                          0
Reported by                    0
Falls within                   0
Longitude                 167326
Latitude                  167326
Location                       0
LSOA code                 167356
LSOA name                 167356
Crime type                     0
Last outcome category    1155292
Context                  8441407
dtype: int64


List of null values in every column 
 Crime ID                 0
Month                    0
Reported by              0
Falls within             0
Longitude                0
Latitude                 0
Location                 0
LSOA code                0
LSOA name                0
Crime type               0
Last outcome category    0
dtype: int64



In [8]:
# Dropping "Falls within" column
masterDf = masterDf.drop("Falls within", axis='columns')

for col in masterDf:
    print(col)

Crime ID
Month
Reported by
Longitude
Latitude
Location
LSOA code
LSOA name
Crime type
Last outcome category


In [9]:
# Clean Force names to be just the name of the police force without "Force" or other keywords
masterDf['Force'] = masterDf['Reported by'].str.replace("Police", "").str.strip()
# Clean metropolitan
masterDf.loc[masterDf['Force'].str.contains('Metropolitan', case=False), 'Force'] = 'Metropolitan'
masterDf['Force'] = masterDf['Force'].str.title()

print(masterDf['Reported by'])

masterDf = masterDf.drop('Reported by', axis='columns')

for col in masterDf:
    print(col)

0          Metropolitan Police Service
1          Metropolitan Police Service
2          Metropolitan Police Service
3          Metropolitan Police Service
4          Metropolitan Police Service
                      ...             
9453120           Thames Valley Police
9453121           Thames Valley Police
9453122           Thames Valley Police
9453123           Thames Valley Police
9453124           Thames Valley Police
Name: Reported by, Length: 8441407, dtype: object
Crime ID
Month
Longitude
Latitude
Location
LSOA code
LSOA name
Crime type
Last outcome category
Force


In [10]:
# Standardize 'Crime type' and 'Last outcome category'
masterDf['Crime type'] = masterDf['Crime type'].str.strip().str.capitalize()
masterDf['Last outcome category'] = masterDf['Last outcome category'].str.strip().str.capitalize()
print(masterDf.head())

                                            Crime ID    Month  Longitude  \
0  a8977a2a4e14252420371eb993d52e4d0b8288a7c833e6...  2019-01  -0.709911   
1                                        No Crime ID  2019-01   0.140192   
2  934e173f2bc2e1dd3a257b37939d8f97575d3eeb89ff0c...  2019-01   0.140192   
3  4f5b7e424bc78b1fb8c32e07da61176d2cbc5a3849d8e1...  2019-01   0.140634   
4  53d960600a4a9f54b785f598af4c75bdef2f79bce1a41b...  2019-01   0.141143   

    Latitude                     Location  LSOA code  \
0  50.784615     On or near Rochester Way  E01031384   
1  51.582311       On or near Hatch Grove  E01000027   
2  51.582311       On or near Hatch Grove  E01000027   
3  51.583427        On or near Rams Grove  E01000027   
4  51.590873  On or near Furze Farm Close  E01000027   

                   LSOA name                    Crime type  \
0                  Arun 018E  Violence and sexual offences   
1  Barking and Dagenham 001A         Anti-social behaviour   
2  Barking and Dagen

In [11]:

yearConversion = pd.to_datetime(masterDf['Month'])
masterDf['Year'] = yearConversion.dt.year
    
display(masterDf)

,Crime ID,Month,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Force,Year
0,a8977a2a4e14252420371eb993d52e4d0b8288a7c833e6...,2019-01,-0.709911,50.784615,On or near Rochester Way,E01031384,Arun 018E,Violence and sexual offences,Status update unavailable,Metropolitan,2019
1,No Crime ID,2019-01,0.140192,51.582311,On or near Hatch Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,Unknown outcome category,Metropolitan,2019
2,934e173f2bc2e1dd3a257b37939d8f97575d3eeb89ff0c...,2019-01,0.140192,51.582311,On or near Hatch Grove,E01000027,Barking and Dagenham 001A,Burglary,Status update unavailable,Metropolitan,2019
3,4f5b7e424bc78b1fb8c32e07da61176d2cbc5a3849d8e1...,2019-01,0.140634,51.583427,On or near Rams Grove,E01000027,Barking and Dagenham 001A,Burglary,Status update unavailable,Metropolitan,2019
4,53d960600a4a9f54b785f598af4c75bdef2f79bce1a41b...,2019-01,0.141143,51.590873,On or near Furze Farm Close,E01000027,Barking and Dagenham 001A,Drugs,Investigation complete; no suspect identified,Metropolitan,2019
...,...,...,...,...,...,...,...,...,...,...,...
9453120,1a290c6bed646acafa5ae267878c9e74ad363ab04a4de3...,2024-12,0.000000,0.000000,No Location,No LSOA Code assigned,No LSOA Name assigned,Other crime,Under investigation,Thames Valley,2024
9453121,b66a5c6320476a6a6fae6f3d8600e7a9c39a780cee7874...,2024-12,0.000000,0.000000,No Location,No LSOA Code assigned,No LSOA Name assigned,Other crime,Under investigation,Thames Valley,2024
9453122,cb80080e0164fa9d12767bcfe912ed902b1bc3db0ce83d...,2024-12,0.000000,0.000000,No Location,No LSOA Code assigned,No LSOA Name assigned,Other crime,Under investigation,Thames Valley,2024
9453123,c49ac623430308ffb6f815e12725a9d3244c7bf3275592...,2024-12,0.000000,0.000000,No Location,No LSOA Code assigned,No LSOA Name assigned,Other crime,Awaiting court outcome,Thames Valley,2024


In [12]:
crimeAggregation = masterDf.groupby(['Force', 'Year', 'Crime type']).size().reset_index(name='Crime count')
crimeAggregation.head()

,Force,Year,Crime type,Crime count
0,Metropolitan,2019,Anti-social behaviour,129696
1,Metropolitan,2019,Bicycle theft,18536
2,Metropolitan,2019,Burglary,78816
3,Metropolitan,2019,Criminal damage and arson,52598
4,Metropolitan,2019,Drugs,43558


In [13]:
xls = pd.ExcelFile('policeforceareas1991to2024.xlsx')
pop_11_20 = pd.read_excel(xls, 'Mid-2011 to Mid-2020', skiprows=3)
pop_21_24 = pd.read_excel(xls, 'Mid-2021 to Mid-2024', skiprows=3)

popAll = pd.concat([pop_11_20, pop_21_24], axis=0)
popAll.head()




,PFA 2023 Code,PFA 2023 Name,Year,F0,F1,F2,F3,F4,F5,F6,...,M76,M77,M78,M79,M80,M81,M82,M83,M84,M85
0,E23000001,Metropolitan Police,2011,61350,59379,57697,57186,54527,52518,49086,...,16821,15777,14827,14280,13328,12373,10812,9522,8565,40890
1,E23000001,Metropolitan Police,2012,64693,60570,58582,57211,56859,54303,52219,...,17456,16016,15024,14039,13417,12387,11448,9909,8637,42292
2,E23000001,Metropolitan Police,2013,63201,63403,59518,57938,56781,56590,53985,...,17250,16625,15262,14225,13230,12591,11369,10485,9005,43356
3,E23000001,Metropolitan Police,2014,61334,61617,61625,58383,57221,56382,56211,...,17521,16447,15850,14445,13404,12422,11739,10384,9581,45254
4,E23000001,Metropolitan Police,2015,61691,59485,59715,60000,57438,56610,55980,...,17556,16738,15633,14966,13568,12501,11507,10822,9375,46343


In [14]:
popAll = pd.concat([pop_11_20, pop_21_24], axis=0)
populationColumns = popAll.columns[3:]
popAll['Population'] = popAll[populationColumns].sum(axis=1)

popClean = popAll[['PFA 2023 Name', 'Year', 'Population']].copy()
popClean.columns = ['Force', 'Year', 'Population']
popClean = popClean.reset_index(drop=True)

print(popClean.head())

                 Force  Year  Population
0  Metropolitan Police  2011     8196995
1  Metropolitan Police  2012     8313253
2  Metropolitan Police  2013     8431450
3  Metropolitan Police  2014     8539625
4  Metropolitan Police  2015     8651877


In [16]:
popClean['Force'] = popClean['Force'].str.replace("Police", "").str.strip()

print(popClean.head())


          Force  Year  Population
0  Metropolitan  2011     8196995
1  Metropolitan  2012     8313253
2  Metropolitan  2013     8431450
3  Metropolitan  2014     8539625
4  Metropolitan  2015     8651877


In [17]:
finalData = pd.merge(crimeAggregation, popClean, on=['Force', 'Year'], how='left')
finalData.head()

print(masterDf['Force'].unique())

['Metropolitan' 'Surrey' 'Sussex' 'Thames Valley']


In [20]:
force_lookup = {
    # Metropolitan 
    'Barking and Dagenham': 'Metropolitan', 'Barnet': 'Metropolitan', 'Bexley': 'Metropolitan',
    'Brent': 'Metropolitan', 'Bromley': 'Metropolitan', 'Camden': 'Metropolitan',
    'Croydon': 'Metropolitan', 'Ealing': 'Metropolitan', 'Enfield': 'Metropolitan',
    'Greenwich': 'Metropolitan', 'Hackney': 'Metropolitan', 'Hammersmith and Fulham': 'Metropolitan',
    'Haringey': 'Metropolitan', 'Harrow': 'Metropolitan', 'Havering': 'Metropolitan',
    'Hillingdon': 'Metropolitan', 'Hounslow': 'Metropolitan', 'Islington': 'Metropolitan',
    'Kensington and Chelsea': 'Metropolitan', 'Kingston upon Thames': 'Metropolitan',
    'Lambeth': 'Metropolitan', 'Lewisham': 'Metropolitan', 'Merton': 'Metropolitan',
    'Newham': 'Metropolitan', 'Redbridge': 'Metropolitan', 'Richmond upon Thames': 'Metropolitan',
    'Southwark': 'Metropolitan', 'Sutton': 'Metropolitan', 'Tower Hamlets': 'Metropolitan',
    'Waltham Forest': 'Metropolitan', 'Wandsworth': 'Metropolitan', 'Westminster': 'Metropolitan',
    'City of London': 'Metropolitan', 
    # Surrey
    'Elmbridge': 'Surrey', 'Epsom and Ewell': 'Surrey', 'Guildford': 'Surrey',
    'Mole Valley': 'Surrey', 'Reigate and Banstead': 'Surrey', 'Runnymede': 'Surrey',
    'Spelthorne': 'Surrey', 'Surrey Heath': 'Surrey', 'Tandridge': 'Surrey',
    'Waverley': 'Surrey', 'Woking': 'Surrey',

    # Sussex
    'Adur': 'Sussex', 'Arun': 'Sussex', 'Brighton and Hove': 'Sussex',
    'Chichester': 'Sussex', 'Crawley': 'Sussex', 'Eastbourne': 'Sussex',
    'Hastings': 'Sussex', 'Horsham': 'Sussex', 'Lewes': 'Sussex',
    'Mid Sussex': 'Sussex', 'Rother': 'Sussex', 'Wealden': 'Sussex', 'Worthing': 'Sussex',

    # Thames Valley
    'Bracknell Forest': 'Thames Valley', 'Buckinghamshire': 'Thames Valley', 'Cherwell': 'Thames Valley',
    'Milton Keynes': 'Thames Valley', 'Oxford': 'Thames Valley', 'Reading': 'Thames Valley',
    'Slough': 'Thames Valley', 'South Oxfordshire': 'Thames Valley', 'Vale of White Horse': 'Thames Valley',
    'West Berkshire': 'Thames Valley', 'West Oxfordshire': 'Thames Valley',
    'Windsor and Maidenhead': 'Thames Valley', 'Wokingham': 'Thames Valley'
}

iod2025 = pd.read_csv("File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv")
iod2025['Force'] = iod2025['Local Authority District name (2024)'].map(force_lookup)

excludedEntries = iod2025['Force'].isna().sum()
print(f"There are {excludedEntries} out of {len(iod2025)} LSOAs that are outside the 4 forces")

There are 25528 out of 33755 LSOAs that are outside the 4 forces


In [21]:
iodFiltered = iod2025.dropna(subset=['Force']).copy()
imd_final = iodFiltered.groupby('Force').agg({
    'Index of Multiple Deprivation (IMD) Score': 'mean',
    'Total population: mid 2022': 'sum'
}).reset_index()

# 4. Final check - you should have exactly 4 rows now
print(imd_final)

           Force  Index of Multiple Deprivation (IMD) Score  \
0   Metropolitan                                  22.128675   
1         Surrey                                  10.108495   
2         Sussex                                  17.677184   
3  Thames Valley                                  12.673253   

   Total population: mid 2022  
0                     8869043  
1                     1217048  
2                     1722184  
3                     2551808  
